# Project 2 (take-home): Photo Compressor App 📸

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/future-of-mathematicians/notebooks/02_photo_compressor_app.ipynb)

> **From your syllabus**
> - *Linear Algebra*: the singular value decomposition $A=U\Sigma V^\top$
> - *ML-II*, Unit IV: low-rank structure, PCA

**Your project:** a web app where anyone uploads a photo, moves a slider, and sees it rebuilt from only $k$ "SVD layers", with the exact storage saved and error paid. Same recipe as Project 1: build, check, ship, package.

If you did Workshop 1, this is Mission 3 turned into a product.

> 📊 **Progress tracking: please read.** The next cell draws your progress map and records which exercises you finish. It sends your **name, roll number, exercise ID and time** to the instructor's progress sheet for this workshop, nothing else (no files, nothing from your Google account). Its code is collapsed to save space but is all there to read (Colab: the cell's **⋮** menu → *Show code*; JupyterLab: click the collapsed input). Prefer not to be tracked? Type `anonymous` as your name, or set `TRACKING = False`; the map still works for you.
>
> Colab may ask *"Do you trust this code?"* because this notebook comes from GitHub (`anshid/MathMeetsML`, open source, MIT licence). That is expected: choose **Run anyway**.

In [ ]:
#@title Progress map (just run me)
# Records which exercises you finish so you can see your progress map and the instructor can see how the class is doing.
# Sent: your name, roll number, exercise id and time. Nothing else. Type "anonymous" as your name (or set TRACKING = False) to send nothing.
TRACKING = True
MISSION = 2

import json, os, time, base64, urllib.parse, urllib.request
from IPython.display import HTML, display, update_display

CLOUD_URL = "https://script.google.com/macros/s/AKfycbxekbOkIE5oRVA319mhEENS1phVLt7gPfRUernniP8kMopgIqUNHNN37xjF_3XZEuQWwQ/exec"
TOKEN = "3F5y-xKcPbZZPT49"
LAN_URL = os.environ.get("MML_LAN", "")
MISSIONS = {"1": {"title": "Digit App", "checkpoints": ["p01-ready", "p01-first-model", "p01-model", "p01-app", "p01-done"]}, "2": {"title": "Photo App", "checkpoints": ["p02-svd", "p02-ey", "p02-app", "p02-done"]}, "3": {"title": "Risk App", "checkpoints": ["p03-model", "p03-threshold", "p03-app", "p03-done"]}}
LABEL, UNIT = "P", "Project"
ORDER = sorted(MISSIONS, key=int)
STORE = "progress.json"
SINKS = [u for u in (CLOUD_URL, LAN_URL) if u]


def _load():
    try:
        with open(STORE) as f:
            return json.load(f)
    except Exception:
        return {"name": "", "roll": "", "done": {}, "queue": []}


def _save(s):
    try:
        with open(STORE, "w") as f:
            json.dump(s, f)
    except Exception:
        pass


def _who(s):
    if s["roll"]:
        return
    if os.environ.get("MML_TRACKING") == "off":
        s["name"], s["roll"] = "anonymous", "-"
        return
    try:
        s["name"] = input("Your name (or 'anonymous'): ").strip() or "anonymous"
        s["roll"] = input("Roll number: ").strip() or "-"
    except Exception:
        s["name"], s["roll"] = "anonymous", "-"


def _sending(s):
    return TRACKING and s["name"].lower() != "anonymous" and os.environ.get("MML_TRACKING") != "off"


def _post(url, ev):
    body = json.dumps(dict(ev, token=TOKEN)).encode()
    urllib.request.urlopen(urllib.request.Request(url, body, {"Content-Type": "text/plain"}), timeout=2).read()


def sync():
    """Send queued exercises to the instructor. Silent if offline; they stay queued."""
    s = _load()
    if _sending(s):
        dead = set()
        for ev in s["queue"]:
            for u in SINKS:
                if u in dead or u in ev["sent"]:
                    continue
                try:
                    _post(u, ev)
                    ev["sent"].append(u)
                except Exception:
                    dead.add(u)
        s["queue"] = [e for e in s["queue"] if any(u not in e["sent"] for u in SINKS)]
    _save(s)
    return s


def progress_code():
    """Offline? Paste this code to the instructor (chat or USB) instead."""
    s = _load()
    print(base64.b64encode(json.dumps({"name": s["name"], "roll": s["roll"], "done": s["done"]}).encode()).decode())


def _status(s, n):
    ids = MISSIONS[str(n)]["checkpoints"]
    got = sum(1 for c in ids if c in s["done"])
    if got == len(ids):
        return "done", got, len(ids)
    if n == MISSION or got:
        return "now", got, len(ids)
    return "todo", got, len(ids)


def _html(s):
    cols = {"done": "#2e9e5b", "now": "#3b82f6", "todo": "#8a8f98"}
    parts, stats = [], [_status(s, int(k)) for k in ORDER]
    for i, (st, got, tot) in enumerate(stats):
        n, x = int(ORDER[i]), 45 + i * 108
        title = MISSIONS[str(n)]["title"]
        if i:
            ok = stats[i - 1][0] == "done"
            parts.append(f'<line x1="{x - 108 + 20}" y1="40" x2="{x - 20}" y2="40" stroke="{cols["done"] if ok else cols["todo"]}" stroke-width="3" stroke-dasharray="{"0" if ok else "5 4"}"/>')
        if st == "done":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="{cols[st]}"/><path d="M{x - 8} 40 l6 7 l11 -14" stroke="#fff" stroke-width="3.5" fill="none" stroke-linecap="round" stroke-linejoin="round"/>')
        elif st == "now":
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="4"><animate attributeName="r" values="18;22;18" dur="1.8s" repeatCount="indefinite"/></circle><text x="{x}" y="45" text-anchor="middle" font-size="14" font-weight="700" fill="{cols[st]}">{got}/{tot}</text>')
        else:
            parts.append(f'<circle cx="{x}" cy="40" r="18" fill="none" stroke="{cols[st]}" stroke-width="2.5" stroke-dasharray="4 3"/><text x="{x}" y="45" text-anchor="middle" font-size="14" fill="{cols[st]}">{n}</text>')
        parts.append(f'<text x="{x}" y="82" text-anchor="middle" font-size="12" fill="currentColor" opacity="{1 if st != "todo" else 0.55}">{LABEL}{n} {title}</text>')
    finished = [k for k, t in zip(ORDER, stats) if t[0] == "done"]
    todo = [f"{LABEL}{k}" for k, t in zip(ORDER, stats) if t[0] == "todo"]
    now = stats[ORDER.index(str(MISSION))]
    line = f"<b>{len(finished)} of {len(stats)} {UNIT.lower()}s finished</b> &middot; now in <b>{UNIT} {MISSION}</b> ({now[1]}/{now[2]} exercises)"
    line += f" &middot; not visited yet: {', '.join(todo)}" if todo else f" &middot; every {UNIT.lower()} visited"
    who = f"{s['name']} ({s['roll']})" if s["roll"] else ""
    return (f'<div style="font-family:sans-serif;max-width:700px"><svg viewBox="0 0 640 96" width="100%" role="img" aria-label="Progress map">{"".join(parts)}</svg>'
            f'<div style="font-size:14px">{line}</div><div style="font-size:12px;opacity:.6">{who}</div></div>')


_shown = []


def show_tracker():
    html = HTML(_html(_load()))
    if _shown:
        try:
            update_display(html, display_id="mml-tracker")
            return
        except Exception:
            pass
    display(html, display_id="mml-tracker")
    _shown.append(1)


def checkpoint(cid):
    """Call after an exercise passes: saves it, tells the instructor (if online), refreshes the map."""
    s = _load()
    _who(s)
    if cid not in s["done"]:
        s["done"][cid] = time.strftime("%H:%M:%S")
        s["queue"].append({"name": s["name"], "roll": s["roll"], "mission": MISSION, "checkpoint": cid, "ts": s["done"][cid], "sent": []})
    _save(s)
    sync()
    show_tracker()


def _start():
    s = _load()
    _who(s)
    if CLOUD_URL and _sending(s):  # Colab gives every notebook a fresh disk, so ask what you already finished elsewhere
        try:
            q = urllib.parse.urlencode({"roll": s["roll"], "token": TOKEN})
            for c in json.load(urllib.request.urlopen(CLOUD_URL + "?" + q, timeout=2)).get("done", []):
                s["done"].setdefault(c, "earlier")
        except Exception:
            pass
    _save(s)
    show_tracker()


_start()

In [ ]:
#@title ▶ Setup (just run me; code hidden)
import os, sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_images

try:
    import gradio as gr
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio==6.31.0"], check=True)
    import gradio as gr

CHECK = os.environ.get("MML_CHECK") == "1"
photo = load_sample_images().images[0]                  # bundled sample photo (china.jpg, CC-BY 2.0)
img = photo / 255.0
print("photo shape (height, width, colours):", img.shape)
plt.imshow(img); plt.axis("off"); plt.show()

## Stage 1. One colour channel is a matrix
Write the rank-$k$ approximation $A_k=\sum_{i\le k}\sigma_iu_iv_i^\top=U_k\,\mathrm{diag}(s_k)\,V_k^\top$.

<details><summary>Hint</summary>

`(U[:, :k] * s[:k]) @ Vt[:k, :]`
</details>

In [ ]:
def compress_channel(A, k):
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    return ____, s                                    # TODO: the rank-k approximation of A

red = img[..., 0]
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for ax, k in zip(axes, [1, 5, 20, 80]):
    ax.imshow(compress_channel(red, k)[0], cmap="gray"); ax.set_title(f"k = {k}"); ax.axis("off")
plt.show()
checkpoint("p02-svd")

## Stage 2. Check the theorem before you trust the app
> **Eckart-Young.** $A_k$ is the best rank-$k$ approximation and $\|A-A_k\|_F=\sqrt{\sum_{i>k}\sigma_i^2}$.

Fill in the right-hand side of the formula.

In [ ]:
for k in [5, 20, 80]:
    Ak, s = compress_channel(red, k)
    lhs = np.linalg.norm(red - Ak)
    rhs = ____                                        # TODO: sqrt of the sum of the squared singular values after the k-th
    print(f"k={k:<3} |A-A_k|_F = {lhs:.4f}   formula = {rhs:.4f}")
    assert np.isclose(lhs, rhs)
print("Eckart-Young verified on your photo ✔")
checkpoint("p02-ey")

## Stage 3. The app
Colour = three matrices. Compress each, report storage $k(m+n+1)/(mn)$ and the relative error. The function below is the whole app; run it, then try your own photo in the app.

In [ ]:
APP_PY = r'''# app.py - Photo Compressor with the SVD (MathMeetsML project). Run locally: python app.py
import numpy as np
import gradio as gr


def compress_channel(A, k):
    U, s, Vt = np.linalg.svd(A, full_matrices=False)
    return (U[:, :k] * s[:k]) @ Vt[:k, :], s


def compress(photo, k):
    if photo is None:
        return None, ""
    img = np.asarray(photo, dtype=float)[..., :3] / 255.0
    scale = max(img.shape[:2]) / 600                    # keep the SVD fast on big phone photos
    if scale > 1:
        step = int(np.ceil(scale))
        img = img[::step, ::step]
    m, n, _ = img.shape
    k = int(min(k, m, n))
    out, err2, tot2 = np.zeros_like(img), 0.0, 0.0
    for c in range(3):
        out[..., c], s = compress_channel(img[..., c], k)
        err2 += (s[k:] ** 2).sum()
        tot2 += (s ** 2).sum()
    storage = k * (m + n + 1) / (m * n)
    info = f"rank k = {k} | storage {storage:.1%} of the original | relative error {np.sqrt(err2 / tot2):.2%} (Eckart-Young)"
    return (out.clip(0, 1) * 255).astype(np.uint8), info


demo = gr.Interface(fn=compress,
                    inputs=[gr.Image(type="numpy", label="Your photo"), gr.Slider(1, 200, value=20, step=1, label="rank k")],
                    outputs=[gr.Image(label="Compressed"), gr.Textbox(label="What it cost")],
                    title="SVD Photo Compressor",
                    description="Keeps the top k singular layers of each colour channel. Built at the MathMeetsML workshop.",
                    flagging_mode="never")

if __name__ == "__main__":
    demo.launch()'''
exec(APP_PY.replace('if __name__ == "__main__":', "if False:"))     # defines compress() and demo, same code as app.py

out, info = compress((img * 255).astype(np.uint8), 20)
print(info)
plt.imshow(out); plt.axis("off"); plt.show()
if CHECK:
    print("check mode: app built, not launched")
else:
    demo.launch(share=True)
checkpoint("p02-app")

## Stage 4. Package it (GitHub + Hugging Face Spaces)
Edit `YOUR_NAME` in the first cell, then run both cells.

In [ ]:
YOUR_NAME = "Your Name"

In [ ]:
#@title ▶ Write, test and zip the project (just run me; code hidden)
import runpy, shutil
README = f'''---
title: SVD Photo Compressor
emoji: 📸
sdk: gradio
sdk_version: {gr.__version__}
app_file: app.py
license: mit
---
# SVD Photo Compressor

Upload a photo and choose a rank $k$; each colour channel is replaced by its best rank-$k$ approximation
$A_k=\\sum_{{i\\le k}}\\sigma_iu_iv_i^\\top$. The app reports storage $k(m+n+1)/(mn)$ and the exact error
$\\sqrt{{\\sum_{{i>k}}\\sigma_i^2}}$ from the Eckart-Young theorem, which the notebook verifies numerically.

Built by {YOUR_NAME} at the MathMeetsML workshop. Code MIT.
'''
os.makedirs("photo-compressor", exist_ok=True)
for fname, text in {"app.py": APP_PY, "README.md": README, "requirements.txt": "numpy\n"}.items():
    with open(os.path.join("photo-compressor", fname), "w", encoding="utf-8") as f:
        f.write(text.strip() + "\n")
packed = runpy.run_path("photo-compressor/app.py", run_name="packaged")
assert packed["compress"]((img * 255).astype(np.uint8), 10)[0].shape[2] == 3
shutil.make_archive("photo-compressor", "zip", "photo-compressor")
if not CHECK:
    try:
        from google.colab import files
        files.download("photo-compressor.zip")
    except ImportError:
        print("Not on Colab: the zip is next to this notebook.")
print("packaged and tested ✔  ->  photo-compressor.zip")
checkpoint("p02-done")

## Stretch
- Plot the smallest $k$ with relative error under 5% for several photos. Which kinds of photo are "low rank"?
- Show the *error image* $A-A_k$: what detail does the SVD throw away first?

## You just used...
- the **SVD** and the **Eckart-Young theorem**, checked numerically
- the cost-benefit trade-off of low-rank models (storage vs error), the idea behind PCA and LoRA fine-tuning of large AI models

## Your CV line
> *Built and deployed an SVD image-compression web app (NumPy, Gradio) reporting storage and the Eckart-Young error bound, verified numerically.*

Next: **Project 3**, a probability-based risk estimator.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshid/MathMeetsML/blob/main/future-of-mathematicians/notebooks/03_risk_estimator_app.ipynb)